![FMD_Overview](https://github.com/edkreuk/FMD_FRAMEWORK/blob/main/Images/FMD_DOMAIN_OVERVIEW.png?raw=true)

# Bootstrap en manifest

In [ ]:
# =============================================================================
# BOOTSTRAP - waar staat je fork?
# Deze vier regels worden gestempeld door NB_BOOTSTRAP_FMD. Pas ze alleen met de
# hand aan als je dit notebook los importeert.
# =============================================================================
repo_owner    = "bitmetric-fabric"
repo_name     = "FMD_FRAMEWORK"
branch        = "main"
folder_prefix = ""
github_token_key_vault = ""
github_token_secret    = ""

# =============================================================================
# Manifest (ADR-003)
# Alles wat per klant vaststaat en daarna niet meer verandert, staat in
# manifest.yaml in de root van de repo hierboven. Niets daarvan hoort nog in een
# notebookcel. Zie manifest.example.yaml voor de volledige structuur.
# =============================================================================
import requests as _requests
import yaml as _yaml

MANIFEST_URL = f"https://raw.githubusercontent.com/{repo_owner}/{repo_name}/{branch}/manifest.yaml"

github_token = (
    notebookutils.credentials.getSecret(
        f"https://{github_token_key_vault}.vault.azure.net/", github_token_secret
    )
    if github_token_key_vault and github_token_secret
    else None
)
github_headers = {"Authorization": f"Bearer {github_token}"} if github_token else {}

_response = _requests.get(MANIFEST_URL, headers=github_headers)
if _response.status_code == 404:
    raise RuntimeError(
        f"Geen manifest.yaml gevonden op {MANIFEST_URL}\n"
        f"Kopieer manifest.example.yaml naar manifest.yaml in de root van "
        f"{repo_owner}/{repo_name} (branch '{branch}'), vul hem in en commit hem."
    )
_response.raise_for_status()
manifest = _yaml.safe_load(_response.text) or {}


def _lookup(path):
    """Geneste opzoeking op 'a.b.c'. Geeft (waarde, gevonden) terug."""
    node = manifest
    for part in path.split("."):
        if not isinstance(node, dict) or part not in node:
            return None, False
        node = node[part]
    return node, True


# Alles wat dit notebook nodig heeft. In een keer gecontroleerd, zodat je niet
# per ontbrekende sleutel opnieuw hoeft te draaien.
REQUIRED = [
    "repository.owner",
    "repository.name",
    "repository.branch",
    "naming.domain_name",
    "naming.framework_pre_fix",
    "naming.business_domains",
    "environments",
    "framework.lakehouse_schema_enabled",
    "security.admin_group_id",
    "security.contributor_group_id",
    "shortcuts.source_schema",
    "shortcuts.target_schema",
]

_missing = [path for path in REQUIRED if not _lookup(path)[1]]
if _missing:
    raise RuntimeError(
        "manifest.yaml mist verplichte sleutel(s):\n"
        + "\n".join(f"  - {path}" for path in _missing)
        + f"\n\nBron: {MANIFEST_URL}"
        + "\nZie manifest.example.yaml voor de volledige structuur."
    )


def require(path):
    """Verplichte manifest-waarde."""
    value, found = _lookup(path)
    if not found:
        raise KeyError(f"'{path}' staat niet in manifest.yaml ({MANIFEST_URL})")
    return value


def optional(path, default=None):
    """Manifest-waarde die weg mag zijn."""
    value, found = _lookup(path)
    return value if found else default


def principals(*specs):
    """[(id, type), ...] -> principal-dicts. Nul-ID's worden weggelaten: die
    betekenen 'nog niet bekend', en fab acl set loopt erop stuk."""
    out = []
    for principal_id, principal_type in specs:
        if principal_id and str(principal_id).replace("-", "").strip("0"):
            out.append({"id": str(principal_id), "type": principal_type})
    return out


def roles(*specs):
    """[(id, type, role), ...] -> rollijst voor deploy_workspaces."""
    out = []
    for principal_id, principal_type, role in specs:
        for principal in principals((principal_id, principal_type)):
            out.append({"principal": principal, "role": role})
    return out


def ws_suffix(short):
    """Omgevingsmarker voor workspacenamen. Production krijgt er geen: alleen
    Dev/Test/Acceptance worden gemarkeerd, bv. 'INTEGRATION DATA (D)' naast
    'INTEGRATION DATA' in productie."""
    return "" if short == "P" else f" ({short})"


# De bootstrap-cel en het manifest moeten dezelfde repo aanwijzen; anders komen
# src/ en config/ straks ergens anders vandaan dan het manifest beschrijft.
for _key, _bootstrap_value in (
    ("repository.owner", repo_owner),
    ("repository.name", repo_name),
    ("repository.branch", branch),
):
    if str(require(_key)) != str(_bootstrap_value):
        raise RuntimeError(
            f"Bootstrap en manifest spreken elkaar tegen: {_key} is "
            f"'{require(_key)}' in manifest.yaml maar '{_bootstrap_value}' in de "
            f"bootstrap-cel hierboven."
        )

print(f"Manifest geladen van {MANIFEST_URL}")


In [ ]:
%pip install --upgrade ms-fabric-cli pillow cairosvg --quiet

# Configuration and Parameters

**Fabric Administrator Role is required to create domain**

In [ ]:
assign_icons = True                                # Set to True to assign default icons to workspaces; set to False if you have already assigned custom icons

driver = '{ODBC Driver 18 for SQL Server}'          # Change this if you use a different driver
overwrite_variable_library=True                    # By default the Library is overwritten, change this to "False" if you have custom changes

# Uit het manifest - pas deze aan in manifest.yaml, niet hier.
lakehouse_schema_enabled = require('framework.lakehouse_schema_enabled')
spark_version = str(require('spark.runtime_version'))


## KeyVault settings
For future usage

In [ ]:
key_vault_uri_name='val_key_vault_uri_name'
purview_account_name='val_purview_account_name'

## Shortcut settings

In [ ]:
##### Shortcut-schema's komen uit manifest.shortcuts. De workspace- en lakehouse-ID's
##### stonden hier vroeger ook, maar die worden na deployment opgezocht en ingevuld
##### door de cel "Auto-fill VAR_GOLD_SHORTCUTS_FMD" verderop.
SourceSchema          = require('shortcuts.source_schema')          # schema in de Gold-lakehouse
Shortcut_TargetSchema = require('shortcuts.target_schema')          # schema in de Silver-lakehouse


## Capacity configuration

In [ ]:
reassign_capacity= True                                        # If set to False existing assigned capacities to workspaces will no be overwritten

# De capaciteit per omgeving komt uit manifest.environments[].capacity en wordt
# in de cel "Business Domain Configuration" toegepast.


## Domain and Framework settings

In [ ]:
framework_pre_fix= require('naming.framework_pre_fix')  # voorvoegsel voor alle workspace- en pipelinenamen, bv. 'ACME' geeft 'ACME INTEGRATION CODE (D)'
if framework_pre_fix != '':
   framework_pre_fix= framework_pre_fix + ' '           # leeg: geen voorvoegsel; anders met een spatie erachter

##Domains
create_domains=  True                                    # If you do not have a Fabric Admin role, you need to set this option to False. For domain creation the Fabric Admin role is needed

business_domain_names= require('naming.business_domains')  # Define business domains

# Groep die workspaces aan dit domein mag toevoegen of verwijderen.
domain_contributor_role = {"type": "Contributors", "principals": principals(
    (require('security.contributor_group_id'), "Group"),
)}

# Afgeleid van naming.domain_name, net als in NB_SETUP_FMD - niet apart instellen.
configuration_database_workspace = framework_pre_fix + require('naming.domain_name') + ' CONFIG'
configuration_database_name      = 'SQL_' + require('naming.domain_name') + '_FRAMEWORK'


In [ ]:
configuration = {
                    'workspace': {
                        'name' : configuration_database_workspace            # Name of target workspace
                                       },
                    'DatabaseName' : configuration_database_name                    # Name of target configuration SQL Database
}

# Configuration


## Workspace Roles Configuration

In [ ]:
# Rollen komen uit manifest.security. Een ID dat nog op nullen staat betekent
# "nog niet bekend" en wordt overgeslagen; fab acl set loopt daar anders op stuk.
workspace_roles_data_business_domain = roles(
    (require('security.admin_group_id'), 'Group', 'admin'),
)

workspace_roles_code_business_domain = roles(
    (require('security.admin_group_id'), 'Group', 'admin'),
)

workspace_roles_reporting_business_domain = roles(
    (require('security.admin_group_id'), 'Group', 'admin'),
)


## Business Domain Configuration

In [ ]:
##### Opgebouwd uit manifest.environments - voeg omgevingen daar toe, niet hier ####

def business_domain_capacity(env):
    """Business domains mogen op een eigen capaciteit draaien. Staat
    capacity_business_domain niet in het manifest, dan delen ze die van de
    ingestielaag - zoals het voor het manifest ook ging."""
    return env.get('capacity_business_domain') or env['capacity']


business_domain_deployment = [
    {
        'environment_name': env['name'].lower(),                                    # Sleutel in mapping_table; lowercase zoals deploy_item verwacht
        'environment_short': env['short'],                                          # D / T / A / P
        'workspaces': {
            'data':      {'roles': workspace_roles_data_business_domain,      'capacity_name': business_domain_capacity(env)},
            'code':      {'roles': workspace_roles_code_business_domain,      'capacity_name': business_domain_capacity(env)},
            'reporting': {'roles': workspace_roles_reporting_business_domain, 'capacity_name': business_domain_capacity(env)},
            'semantic':  {'roles': workspace_roles_reporting_business_domain, 'capacity_name': business_domain_capacity(env)},
        },
    }
    for env in require('environments')
]
###################################################


## Download source & config files

In [ ]:
import subprocess
import os
from time import sleep, time
import json
import shutil
import re
import requests
import zipfile
import yaml
import struct
import pyodbc
import cairosvg
import base64
import xml.etree.ElementTree as ET

from PIL import Image, ImageDraw, ImageFont
from requests.adapters import HTTPAdapter, Retry
from io import BytesIO
from zipfile import ZipFile 

In [ ]:
def download_folders_as_zip(repo_owner, repo_name, output_zip, branch="main", folders_to_extract=None, remove_folder_prefix="", headers=None):
    if folders_to_extract is None:
        folders_to_extract = []

    # Construct the URL for the GitHub API to download the repository as a zip file
    url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/zipball/{branch}"
    response = requests.get(url, headers=headers or {})
    response.raise_for_status()

    # Ensure the directory for the output zip file exists
    os.makedirs(os.path.dirname(output_zip), exist_ok=True)

    # Create a zip file in memory from GitHub response
    with zipfile.ZipFile(BytesIO(response.content)) as zipf:
        # Open output zip in append mode
        with zipfile.ZipFile(output_zip, 'w') as output_zipf:
            
            for file_info in zipf.infolist():
                for folder in folders_to_extract:
                    folder_path = f"/{folder}" if not folder.startswith("/") else folder
                    if re.sub(r'^.*?/', '/', file_info.filename).startswith(folder_path):
                        file_data = zipf.read(file_info.filename)
                        parts = file_info.filename.split('/')
                        if remove_folder_prefix:
                            parts = [p for p in parts if p != remove_folder_prefix]
                        output_zipf.writestr('/'.join(parts[1:]), file_data)

def uncompress_zip_to_folder(zip_path, extract_to):
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    os.remove(zip_path)

def copy_to_tmp(name):
    shutil.rmtree("./builtin/tmp", ignore_errors=True)
    path2zip = "./builtin/src/src.zip"

    prefixes = [
        f"src/{name}"
    ]

    with ZipFile(path2zip) as archive:
        for prefix in prefixes:
            matched_files = [file for file in archive.namelist() if file.startswith(prefix)]
            if matched_files:
                for file in matched_files:
                    archive.extract(file, "./builtin/tmp")
                return f"./builtin/tmp/{prefix}"  # Return only the first matching prefix

    return None  # Nothing found

# ✅ Combine all folders into one zip
download_folders_as_zip(repo_owner, repo_name, output_zip = "./builtin/src/src.zip", branch = branch, folders_to_extract= [f"{folder_prefix}/src"] , remove_folder_prefix = f"{folder_prefix}", headers = github_headers)
download_folders_as_zip(repo_owner, repo_name, output_zip = "./builtin/config/config.zip", branch = branch, folders_to_extract= [f"{folder_prefix}/config"] , remove_folder_prefix = f"{folder_prefix}", headers = github_headers)
# ✅ Uncompress everything into ./builtin
uncompress_zip_to_folder(zip_path = "./builtin/config/config.zip", extract_to= "./builtin")

mapping_table=[]
tasks=[]


# CLI Login

In [ ]:
# Set environment parameters for Fabric CLI
token = notebookutils.credentials.getToken('pbi')
os.environ['FAB_TOKEN'] = token
os.environ['FAB_TOKEN_ONELAKE'] = token

# Deployment functions

## Load configuration


In [ ]:
base_path = './builtin/'
config_path = os.path.join(base_path, 'config/item_config.yaml')

with open(config_path, 'r') as file:
        config = yaml.safe_load(file)

deploy_order_path = os.path.join(base_path, 'config/item_initial_setup.json')
with open(deploy_order_path, 'r') as file:
        item_initial_setup =json.load(file)

deploy_order_path = os.path.join(base_path, 'config/data_deployment.json')
with open(deploy_order_path, 'r') as file:
        data_deployment =json.load(file)

deploy_order_path = os.path.join(base_path, 'config/item_deployment_code_business_domain.json')
with open(deploy_order_path, 'r') as file:
        item_deployment_code_business_domain =json.load(file)

deploy_order_path = os.path.join(base_path, 'config/item_deployment_data_business_domain.json')
with open(deploy_order_path, 'r') as file:
        item_deployment_data_business_domain =json.load(file)

deploy_icons_path = os.path.join(base_path, 'config/fabric_icons.xml')

# Parse the XML file
tree = ET.parse(deploy_icons_path)
root = tree.getroot()

# Create a dictionary to store icon name and base64
fabric_icons_fmd = {}
for item in root.findall('icon'):
    name = item.find('name').text if item.find('name') is not None else "No name"
    base64_str = item.find('base64').text if item.find('base64') is not None else ""
    fabric_icons_fmd[name] = base64_str


# Deployment

## Load latest version NB_UTILITIES_SETUP_FMD

In [ ]:
def run_fab_command(command, capture_output=False, silently_continue=False, raw_output=False):
    """
    Executes a Fabric CLI command with optional output capture and error handling.
    """
    result = subprocess.run(["fab", "-c", command], capture_output=capture_output, text=True)
    if not silently_continue and (result.returncode > 0 or result.stderr):
        raise Exception(f"Error running fab command. exit_code: '{result.returncode}'; stderr: '{result}'")
    if capture_output:
        return result if raw_output else result.stdout.strip()
    return None

In [ ]:
workspace_id = notebookutils.runtime.context.get('currentWorkspaceId')
result=run_fab_command("api -X get workspaces/"+workspace_id, capture_output=True, silently_continue=False)
workspace_name=json.loads(result)["text"]["displayName"]

for it in item_initial_setup:
    name = it["name"]
    type = it["type"]
    tmp_path = copy_to_tmp(name)

    cli_parameter = ''

    if "Notebook" in name:
        cli_parameter += " --format .py"
        result = run_fab_command(f"import {workspace_name}.Workspace/{name} -i {tmp_path} -f {cli_parameter}",capture_output=False, silently_continue=True)


In [ ]:
%run NB_UTILITIES_SETUP_FMD

In [ ]:
# VAR_GOLD_SHORTCUTS_FMD declareert zes variabelen in
# config/item_deployment_code_business_domain.json, en update_variable_library zoekt
# elke bron op in variable_parameters. Zonder deze regels faalt het deployen van die
# library met een KeyError die door deploy_item wordt opgevangen en alleen als een
# rood kruisje voorbijkomt - de library wordt dan nooit aangemaakt.
#
# De vier ID's krijgen hier een lege waarde: ze zijn pas bekend als de workspaces en
# lakehouses bestaan. De cel "Auto-fill VAR_GOLD_SHORTCUTS_FMD" zoekt ze verderop op
# en schrijft ze in de gedeployde library.
variable_parameters.update({
    "SourceWorkspaceId": "",
    "SourceLakehouseId": "",
    "SourceSchema": SourceSchema,
    "Shortcut_TargetSchema": Shortcut_TargetSchema,
    "Shortcut_TargetWorkspaceId": "",
    "Shortcut_TargetLakehouseId": "",
})


## Integration Domain

### Get Fabric database (Configuration)

In [ ]:
for deployment_item in data_deployment:
        if deployment_item['type'] in ('SQLDatabase'):
                name = configuration['DatabaseName']
                type = deployment_item["type"]
                name=name+'.'+type
                server, database_name=deploy_item(workspace_name=configuration['workspace']['name'],name=name,mapping_table=mapping_table, environment_name='config',tasks= tasks, lakehouse_schema_enabled=lakehouse_schema_enabled,it=deployment_item)

##!NOTE: in Case of an error check if you're not using a trial capacitiy or the capacity you assigned is paused

## Business Domain

### Create workspaces(Business Domains)

In [ ]:
for business_domain_name in business_domain_names:
    if create_domains:
        create_fabric_domain(business_domain_name)

    for business_domain in business_domain_deployment:
        print(f"--------------------------")
        print(f"Updating Workspace: {business_domain['environment_name']}")
        deploy_workspaces(business_domain_name,workspace=business_domain['workspaces']['code'],workspace_name=framework_pre_fix + business_domain_name + ' CODE'+ws_suffix(business_domain['environment_short']),  environment_name=business_domain['environment_name'], old_id=config["workspaces"]["workspace_business_domain_code"], mapping_table=mapping_table, tasks=tasks)
        deploy_workspaces(business_domain_name,workspace=business_domain['workspaces']['data'],workspace_name=framework_pre_fix + business_domain_name + ' DATA'+ws_suffix(business_domain['environment_short']),  environment_name=business_domain['environment_name'], old_id=config["workspaces"]["workspace_business_domain_data"], mapping_table=mapping_table, tasks=tasks)
        deploy_workspaces(business_domain_name,workspace=business_domain['workspaces']['reporting'],workspace_name=framework_pre_fix + business_domain_name + ' REPORTING'+ws_suffix(business_domain['environment_short']), environment_name=business_domain['environment_name'], old_id=config["workspaces"]["workspace_business_domain_reporting"], mapping_table=mapping_table, tasks=tasks)
        deploy_workspaces(business_domain_name,workspace=business_domain['workspaces']['semantic'],workspace_name=framework_pre_fix + business_domain_name + ' SEMANTIC'+ws_suffix(business_domain['environment_short']), environment_name=business_domain['environment_name'], old_id=config["workspaces"]["workspace_business_domain_reporting"], mapping_table=mapping_table, tasks=tasks)

### Create Lakehouses(Business Domains)

In [ ]:
for workspace_business_domain in business_domain_deployment:
    for business_domain in business_domain_names:
        for workspace in [workspace_business_domain['workspaces']['data']]:
            exclude = []
            for it in item_deployment_data_business_domain:
                new_id = None                    
                name = it["name"]
                type = it["type"]
                if name in exclude:
                    continue
                deploy_item(framework_pre_fix + business_domain + ' DATA'+ws_suffix(workspace_business_domain['environment_short']), name,mapping_table, workspace_business_domain['environment_name'], tasks, lakehouse_schema_enabled,it)

### Create Items(Business Domains)

In [ ]:
for workspace_business_domain in business_domain_deployment:
    for business_domain_name in business_domain_names:
        for workspace in [workspace_business_domain['workspaces']['code']]:
                for it in item_deployment_code_business_domain:
                    new_id = None
                    if it['type'] in ('VariableLibrary'):
                        name = it["name"]
                        type = it["type"]
                        deploy_item(framework_pre_fix + business_domain_name + ' CODE'+ws_suffix(workspace_business_domain['environment_short']), name,mapping_table, workspace_business_domain['environment_name'], tasks, lakehouse_schema_enabled,it)

### Auto-fill VAR_GOLD_SHORTCUTS_FMD (Gold <-> Silver workspace/lakehouse IDs)

In [ ]:
##### Resolves the Gold lakehouse (this business domain) and Silver lakehouse (integration domain) IDs #####
##### per stage, and writes them into VAR_GOLD_SHORTCUTS_FMD so NB_CREATE_SHORTCUTS doesn't need manual IDs. #####
##### The first stage's IDs become the defaults, the other stages get theirs as value set overrides and each #####
##### CODE workspace activates its own value set, so a deployment pipeline can promote the library safely. #####
##### SourceSchema / Shortcut_TargetSchema still come from the "Shortcut settings" cell above (naming convention, not per-domain). #####
integration_workspace_prefix = require('naming.domain_name')
gold_lakehouse_name = "LH_GOLD_LAYER"
silver_lakehouse_name = "LH_SILVER_LAYER"
value_set_names = {env['short']: env['name'] for env in require('environments')}

for business_domain_name in business_domain_names:
    stage_workspaces = []
    stage_values = {}
    for bd in business_domain_deployment:
        short = bd['environment_short']
        gold_workspace_name = f"{framework_pre_fix}{business_domain_name} DATA{ws_suffix(short)}"
        silver_workspace_name = f"{framework_pre_fix}{integration_workspace_prefix} DATA{ws_suffix(short)}"
        code_workspace_name = f"{framework_pre_fix}{business_domain_name} CODE{ws_suffix(short)}"

        gold_workspace_id = get_workspace_id_by_name(gold_workspace_name)
        gold_lakehouse_id = get_item_id(gold_workspace_name, f"{gold_lakehouse_name}.Lakehouse", "id")
        silver_workspace_id = get_workspace_id_by_name(silver_workspace_name)
        silver_lakehouse_id = get_item_id(silver_workspace_name, f"{silver_lakehouse_name}.Lakehouse", "id")

        if not (gold_workspace_id and gold_lakehouse_id and silver_workspace_id and silver_lakehouse_id):
            # Empty rather than skipped: a skipped stage would inherit the first stage's IDs
            print(f"❌ Could not resolve Gold/Silver lakehouse IDs for '{business_domain_name}' ({short}), they stay empty")

        stage = value_set_names[short]
        stage_workspaces.append((stage, code_workspace_name))
        stage_values[stage] = {
            "SourceWorkspaceId": gold_workspace_id or "",
            "SourceLakehouseId": gold_lakehouse_id or "",
            "Shortcut_TargetWorkspaceId": silver_workspace_id or "",
            "Shortcut_TargetLakehouseId": silver_lakehouse_id or "",
        }

    set_variable_library_stage_values("VAR_GOLD_SHORTCUTS_FMD", stage_workspaces, stage_values)

In [ ]:
for workspace_business_domain in business_domain_deployment:
    for business_domain_name in business_domain_names:
        for workspace in [workspace_business_domain['workspaces']['code']]:
                for it in item_deployment_code_business_domain:
                    new_id = None
                    if it['type'] not in ('VariableLibrary'):
                        name = it["name"]
                        type = it["type"]
                        deploy_item(framework_pre_fix + business_domain_name + ' CODE'+ws_suffix(workspace_business_domain['environment_short']), name,mapping_table, workspace_business_domain['environment_name'], tasks, lakehouse_schema_enabled,it)

### Create Deployment Pipelines (Business Domains)

In [ ]:
for business_domain_name in business_domain_names:
    for group in ["code", "data", "reporting", "semantic"]:
        pipeline_name = f"{framework_pre_fix}{business_domain_name} {group.upper()}"
        stage_workspace_names = [
            (bd["environment_name"].capitalize(),
             framework_pre_fix + business_domain_name + " " + group.upper() + ws_suffix(bd["environment_short"]))
            for bd in business_domain_deployment
        ]
        deploy_deployment_pipeline(pipeline_name, stage_workspace_names)

### Wire Business Domains into PL_FMD_ORCHESTRATION_TEMPLATE

In [ ]:
##### Adds an InvokePipeline activity to the integration domain's orchestrator per business domain and stage. #####
##### Skips a domain/stage if it's already wired in (idempotent). #####
for business_domain_name in business_domain_names:
    for bd in business_domain_deployment:
        orchestration_workspace_name = f"{framework_pre_fix}{integration_workspace_prefix} CODE{ws_suffix(bd['environment_short'])}"
        gold_workspace_name = f"{framework_pre_fix}{business_domain_name} CODE{ws_suffix(bd['environment_short'])}"
        activity_name = f"PL_{business_domain_name}_LOAD_GOLD"
        wire_domain_into_orchestration(orchestration_workspace_name, gold_workspace_name, activity_name)

## Workspace Icons

In [ ]:
if assign_icons:  
    seen = set()
    workspaces = []
    # Get cluster URL for use in metadata endpoints
    cluster_base_url = get_cluster_url()

    for item in mapping_table:
        if item['ItemType'] == 'Workspace':
            key = (item['Description'], item['new_id'])
            if key not in seen:
                seen.add(key)
                workspaces.append({'displayName': item['Description'], 'id': item['new_id']})
    fabric_icons = fabric_icons_fmd 

    for workspace in workspaces:
        display_name = workspace['displayName'].lower()
       
        # Assign icon
        for icon_key, icon_value in workspace_icon_def['icons'].items():
            if icon_key in display_name:
                workspace["icon"] = icon_value
                workspace_icon = fabric_icons.get(icon_value)
                break
        else:
            workspace["icon"] = None
            workspace_icon = None

        workspace["icon_base64img"] = workspace_icon

In [ ]:
if assign_icons:
    # Dry run - Display pre and post icons based on specified workspace filters and workspace icon definition. Will NOT update any icons!
    display_workspace_icons(workspaces)
    sleep(2)

### Deploy Workspace Icons

In [ ]:
if assign_icons:
    for workspace in workspaces:
            set_workspace_icon(workspace.get('id'), workspace.get('icon_base64img'))

## Create SQL deployment Manifest

### Add Workspaces to Fabric Database

In [ ]:
#add all created workspace to database
custom_sql_deployment = {"queries_stored_procedures": []}
unique_items = {}
for item in mapping_table:
    if item.get("ItemType") == "Workspace":
        unique_items[item["new_id"]] = item["Description"]

# Convert to list of tuples or dicts
workspaces = [{"Description": desc, "new_id": nid} for nid, desc in unique_items.items()]

for workspace in workspaces:
    print(f'EXEC [integration].[sp_UpsertWorkspace](@WorkspaceId = "{workspace["new_id"]}" ,@Name = "{workspace["Description"]}")')
    custom_sql_deployment["queries_stored_procedures"].append(f'EXEC [integration].[sp_UpsertWorkspace] @WorkspaceId = "{workspace["new_id"]}", @Name = "{workspace["Description"]}"')

In [ ]:
for workspace_business_domain in business_domain_deployment:
    for business_domain_name in business_domain_names:
        for workspace in [workspace_business_domain['workspaces']['code']]:
                workspace_id = get_workspace_id_by_name(framework_pre_fix + business_domain_name + ' CODE'+ws_suffix(workspace_business_domain['environment_short']))
                result = run_fab_command(f"api -X get workspaces/{workspace_id}/items", capture_output=True, silently_continue=True)
                existing_items = json.loads(result)['text']
                for item in existing_items.get('value', []):
                    if item['type'] == 'DataPipeline':
                        print(f'EXEC [integration].[sp_UpsertPipeline] @PipelineId = "{item["id"]}", @WorkspaceId = "{workspace_id}" ,@Name = "{item["displayName"]}"')
                        custom_sql_deployment["queries_stored_procedures"].append(f'EXEC [integration].[sp_UpsertPipeline] @PipelineId = "{item["id"]}", @WorkspaceId = "{workspace_id}" ,@Name = "{item["displayName"]}"')

### Add Lakehouses to Fabric Database

In [ ]:
for workspace_business_domain in business_domain_deployment:
    for business_domain_name in business_domain_names:
        for workspace in [workspace_business_domain['workspaces']['data']]:
                workspace_id = get_workspace_id_by_name(framework_pre_fix + business_domain_name + ' DATA'+ws_suffix(workspace_business_domain['environment_short']))
                result = run_fab_command(f"api -X get workspaces/{workspace_id}/items", capture_output=True, silently_continue=True)
                existing_items = json.loads(result)['text']
                for item in existing_items.get('value', []):
                    if item['type'] == 'Lakehouse':
                        print(f'EXEC [integration].[sp_UpsertLakehouse] @LakehouseId = "{item["id"]}", @WorkspaceId = "{workspace_id}" ,@Name = "{item["displayName"]}"')
                        custom_sql_deployment["queries_stored_procedures"].append(f'EXEC [integration].[sp_UpsertLakehouse] @LakehouseId = "{item["id"]}", @WorkspaceId = "{workspace_id}" ,@Name = "{item["displayName"]}"')
                        

## Deploy SQL Code

In [ ]:
#Make sure out token is still valid before we continou, sometime it can take a bit longer that's we will check it again
token = notebookutils.credentials.getToken('pbi')
os.environ['FAB_TOKEN'] = token
os.environ['FAB_TOKEN_ONELAKE'] = token

In [ ]:
for deployment_item in data_deployment:
        if deployment_item['type'] in ('SQLDatabase'):
                name = configuration['DatabaseName']
                type = deployment_item["type"]
                name=name+'.'+type
                if not server:         #it was already set, just in case
                    server=get_item_id(configuration['workspace']['name'], name, 'properties.serverFqdn')
                if not database_name:    
                    database_name=get_item_id(configuration['workspace']['name'], name, 'properties.databaseName')

try:
    i = 0

    token = notebookutils.credentials.getToken('pbi').encode('utf-16-le')
    token_struct = struct.pack(f'<I{len(token)}s', len(token), token)

    print(f"DRIVER={driver};SERVER={server};PORT=1433;DATABASE={database_name};")
    connection = pyodbc.connect(f"DRIVER={driver};SERVER={server};DATABASE={database_name};", attrs_before={1256:token_struct}, timeout=12)

    with connection.cursor() as cursor:
        cursor.execute("SELECT 1")  # Execute the warm-up query (a simple query like 'SELECT 1' can be used)
        cursor.fetchone()
        connection.timeout = 10  # Setting a lower timeout for subsequent queries

    for i, query in enumerate(custom_sql_deployment["queries_stored_procedures"]):
        print(f' - execute "{query}"')
        cursor.execute(query)
        cursor.commit()


    tasks.append({"task_name":f"{workspace.get('displayName')} {database_name} query {i}", "task_duration": 1, "status": f"success"})
except pyodbc.OperationalError as e:
    print(e) 
    tasks.append({"task_name":f"{workspace.get('displayName')} {database_name} query {i}", "task_duration": 1, "status": f"pyodbc failed: {e}"})
except Exception as e:
    print(e) 
    tasks.append({"task_name":f"{workspace.get('displayName')} {database_name} query {i}", "task_duration": 1, "status": f"failed: {e}"})

In [ ]:
display(tasks)